✓ Develop a linear regression model to predict house price based on features such as the
number of rooms, location, size and other relevant factors. Collect a suitable dataset from
Kaggle, preprocess it, and train the model to make accurate predictions.

In [30]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error


In [3]:
df = pd.read_csv("house_prices[1].csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  object 
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  object 
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long  

In [7]:
model = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for x_train_index, x_test_index in model.split(df, df["condition"]):
  x_train = df.loc[x_train_index]
  x_test = df.loc[x_test_index]

In [14]:
ish = pd.DataFrame(x_train)
ish.to_csv("house_train.csv",index= False)

ish_1 = pd.DataFrame(x_test)
ish_1.to_csv("house_test.csv",index= False)


In [15]:
x_train_labels = x_train["price"].copy()
x_train = x_train.drop("price", axis=1)


In [16]:
x_test_labels = x_test["price"].copy()
x_test = x_test.drop("price", axis=1)


In [17]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore'))
])
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, ['id', 'bedrooms', 'bathrooms', 'sqft_living','sqft_lot', 'floors', 'view', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode','lat', 'long', 'sqft_living15', 'sqft_lot15']),
    ("cat", cat_pipeline, ['date','waterfront', 'condition'])
])
x_train_prepared = full_pipeline.fit_transform(x_train)
x_test_prepared = full_pipeline.transform(x_test)

In [40]:
rand =  XGBRegressor(
    n_estimators=600,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
rand.fit(x_train_prepared, x_train_labels)
prediction = rand.predict(x_test_prepared)
rmse = root_mean_squared_error(x_test_labels, prediction)
print(f"RMSE: {rmse:.2f}")

RMSE: 116432.41
